> **Edición pública anonimizada.** Nombres y rutas comerciales se sustituyeron por etiquetas genéricas; no se incluyen bases ni modelos originales. Los resultados históricos se consultan en la tesis adaptada. La demostración pública usa datos y modelos sintéticos.


In [ ]:
from pathlib import Path
import os, sys

# Find the repository independently of the computer or clone location.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / ".tesis-portafolio").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Abre Jupyter desde la carpeta de este repositorio.")
os.chdir(REPO_ROOT / '01 Limpieza Código')
sys.path.insert(0, str(REPO_ROOT / "01 Limpieza Código"))


# Pipeline reproducible de preparación de datos
## Apéndice de tesis — Limpieza de datos de leads e ingeniería de variables

Este notebook documenta y coordina el pipeline completo de preparación de datos utilizado en la tesis. Su enfoque es **priorizar la documentación**: los scripts de Python ubicados en `scripts/` permanecen como la fuente de verdad, mientras que este notebook los ejecuta en secuencia y resume los artefactos que generan.

> **Regla de reproducibilidad:** ejecute este notebook desde la raíz de `01 Limpieza Código` mediante **Kernel → Restart Kernel and Run All Cells**. Todas las rutas son relativas a la raíz del proyecto.

### Orden del pipeline

1. Comprobaciones del entorno
2. Armonización de nuevas exportaciones
3. Construcción del dataset raw final
4. Limpieza del dataset final
5. Construcción de variables finales
6. Validación de los outputs finales
7. Ejecución del análisis exploratorio de datos (EDA) final

Los outputs destinados al modelado excluyen deliberadamente las variables posteriores al resultado que podrían provocar fuga de información (*target leakage*). Se conservan archivos paralelos con dichas variables únicamente para auditoría y trazabilidad.


## Cómo leer este apéndice

Cada etapa contiene cuatro apartados explícitos: **Propósito**, **Inputs**, **Procesamiento realizado** y **Outputs generados**. A continuación, la celda de código ejecuta el script correspondiente como un proceso de Python independiente. Esto conserva el mismo contexto de ejecución que el pipeline de línea de comandos y evita que el estado del notebook modifique silenciosamente la lógica de producción.

Las celdas de diagnóstico no reconstruyen ningún output. Cargan los archivos escritos por los scripts y muestran dimensiones, valores faltantes, duplicados, balance de la variable objetivo, distribuciones categóricas y ejemplos seleccionados. Si un script fuente falla, la ejecución se detiene inmediatamente para impedir que resúmenes antiguos se interpreten como resultados actuales.


## Configuración del notebook

Las utilidades siguientes se limitan a la coordinación y presentación de resultados. No reproducen reglas de limpieza, definiciones de variables, uniones ni lógica de validación de los scripts fuente. Únicamente se emplean la biblioteca estándar de Python, pandas, openpyxl y las utilidades de visualización de Jupyter.


In [ ]:
from pathlib import Path
from datetime import datetime
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, display
from scripts.schema_contract import (
    FORBIDDEN_ANALYTICAL_COLUMNS,
    REQUIRED_ANALYTICAL_LABEL_COLUMNS,
    validate_analytical_label_schema,
)

PROJECT_ROOT = Path.cwd()
SCRIPTS_DIR = Path("scripts")
VENV_PYTHON = PROJECT_ROOT / ".venv" / "bin" / "python"
PIPELINE_PYTHON = VENV_PYTHON if VENV_PYTHON.exists() else Path(sys.executable)

required_scripts = [
    "check_environment.py",
    "01_harmonize_new_exports.py",
    "01b_harmonize_pt3_exports.py",
    "02_build_final_raw_dataset.py",
    "03_clean_final_dataset.py",
    "04_build_final_features.py",
    "05_validate_final_outputs.py",
    "06_run_final_eda.py",
]
missing_scripts = [name for name in required_scripts if not (SCRIPTS_DIR / name).is_file()]
if missing_scripts:
    raise RuntimeError(
        "El notebook debe ejecutarse desde la raíz de '01 Limpieza Código'. "
        f"Scripts faltantes: {missing_scripts}. Directorio actual: {PROJECT_ROOT}"
    )

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)
print(f"Raíz del proyecto: {PROJECT_ROOT}")
print(f"Ejecutable del notebook: {sys.executable}")
print(f"Ejecutable del pipeline: {PIPELINE_PYTHON}")
print(f"Ejecución iniciada: {datetime.now():%Y-%m-%d %H:%M:%S}")

In [ ]:
def run_script(script_name):
    """Ejecuta un script canónico del pipeline desde la raíz del proyecto."""
    script_path = SCRIPTS_DIR / script_name
    display(Markdown(f"**Ejecutando la fuente de verdad:** `{script_path.as_posix()}`"))
    completed = subprocess.run(
        [str(PIPELINE_PYTHON), str(script_path)],
        cwd=PROJECT_ROOT,
        check=True,
    )
    print(f"Finalizado con código de salida {completed.returncode}.")


def load_csv(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"No se encontró el artefacto esperado del pipeline: {path}")
    return pd.read_csv(path, low_memory=False)


def dataset_summary(df, name, id_candidates=("id_trato", "id_Trato")):
    id_col = next((col for col in id_candidates if col in df.columns), None)
    duplicate_rows = int(df.duplicated().sum())
    duplicate_ids = int(df[id_col].duplicated().sum()) if id_col else pd.NA
    overview = pd.DataFrame(
        {
            "métrica": ["filas", "variables", "celdas faltantes", "celdas faltantes (%)", "filas duplicadas", f"valores duplicados de {id_col or 'ID'}"],
            "valor": [
                len(df),
                df.shape[1],
                int(df.isna().sum().sum()),
                round(df.isna().sum().sum() / max(df.size, 1) * 100, 2),
                duplicate_rows,
                duplicate_ids,
            ],
        }
    )
    display(Markdown(f"### {name}"))
    display(overview)

    missing = df.isna().sum().sort_values(ascending=False)
    missing = missing[missing.gt(0)].head(15).rename("faltantes").to_frame()
    if not missing.empty:
        missing["porcentaje"] = (missing["faltantes"] / len(df) * 100).round(2)
        display(Markdown("**Variables con más valores faltantes (15 principales)**"))
        display(missing)
    else:
        display(Markdown("**Valores faltantes:** no se detectaron valores faltantes."))


def show_distribution(df, column, top=20):
    if column not in df.columns:
        print(f"Columna no disponible: {column}")
        return
    counts = df[column].value_counts(dropna=False).head(top).rename("cantidad").to_frame()
    counts["porcentaje"] = (counts["cantidad"] / len(df) * 100).round(2)
    display(Markdown(f"**Distribución: `{column}`**"))
    display(counts)


def artifact_inventory(paths):
    rows = []
    for item in map(Path, paths):
        rows.append(
            {
                "artefacto": item.as_posix(),
                "existe": item.is_file(),
                "tamaño_kb": round(item.stat().st_size / 1024, 1) if item.is_file() else pd.NA,
                "modificado": datetime.fromtimestamp(item.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S") if item.is_file() else pd.NA,
            }
        )
    result = pd.DataFrame(rows)
    display(result)
    return result


def show_markdown_report(path, max_characters=12000):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    text = path.read_text(encoding="utf-8")
    suffix = "\n\n*Vista previa truncada en el notebook.*" if len(text) > max_characters else ""
    display(Markdown(text[:max_characters] + suffix))


def show_excel_sheet(path, sheet_name, rows=10):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    display(pd.read_excel(path, sheet_name=sheet_name).head(rows))

# 1. Comprobaciones del entorno

### Propósito
Confirmar que los paquetes de Python requeridos y la estructura de directorios estén disponibles antes de modificar cualquier dato. Fallar anticipadamente es importante en un pipeline reproducible porque evita generar outputs parcialmente actualizados.

### Inputs
El entorno activo de Python, `requirements.txt`, el directorio `scripts/` y los directorios esperados de datos raw.

### Procesamiento realizado
El script canónico de comprobación del entorno verifica pandas, NumPy, openpyxl y las carpetas principales del proyecto. Después, el notebook inventaría los archivos raw sin analizarlos ni transformarlos.

### Outputs generados
No se genera ningún dataset. El resultado visible en consola y el inventario de archivos constituyen el registro previo a la ejecución.


In [ ]:
run_script("check_environment.py")

raw_inputs = sorted(Path("data/raw").glob("Leads UVG *.xlsx"))
raw_inputs += sorted(Path("data/raw/data_pt2").glob("Leads_*_UVG.csv"))
raw_inputs += [
    Path("data/raw/Base Historica por Producto - fuente_catalogo_a.xlsx"),
    Path("data/raw/Proyectos fuente_catalogo_b-2024.xlsx"),
]
preflight = artifact_inventory(raw_inputs)
if not preflight["existe"].all():
    missing = preflight.loc[~preflight["existe"], "artefacto"].tolist()
    raise FileNotFoundError(f"Faltan inputs raw obligatorios: {missing}")

# 2. Armonización de nuevas exportaciones

### Propósito
Estandarizar las exportaciones de proyectos más recientes antes de incorporarlas a las fuentes históricas en Excel. La armonización resuelve diferencias de esquema y etiquetas en el límite de ingestión, lo que reduce el riesgo de tratar variables equivalentes como columnas distintas.

### Inputs
Archivos delimitados por punto y coma que coinciden con `data/raw/data_pt2/Leads_*_UVG.csv`.

### Procesamiento realizado
`01_harmonize_new_exports.py` identifica las exportaciones y homologa cada par bajo el nombre `*_label`: prioriza el valor label cuando existe y completa sus nulos con la columna original; si un archivo solo contiene la columna original, la renombra al nombre label canónico antes de la unión. También elimina `tipomodelo_label`, normaliza nombres clave del esquema y etiquetas de proyecto, conserva el identificador de la fuente original y valida la conservación de filas y la presencia de etapas requeridas.

### Outputs generados
Archivos CSV armonizados por proyecto, `pt2_all_harmonized.csv` y un reporte en Excel que documenta los reemplazos de etiquetas y su cobertura.


In [ ]:
run_script("01_harmonize_new_exports.py")

In [ ]:
pt2_path = Path("data/raw/data_pt2/harmonized/pt2_all_harmonized.csv")
pt2 = load_csv(pt2_path)
dataset_summary(pt2, "Nuevas exportaciones armonizadas")
show_distribution(pt2, "Proyecto")
show_distribution(pt2, "Etapa Trato")

harmonization_outputs = sorted(Path("data/raw/data_pt2/harmonized").glob("*.csv"))
harmonization_outputs.append(Path("reports/final/label_replacement_report.xlsx"))
artifact_inventory(harmonization_outputs)
show_excel_sheet("reports/final/label_replacement_report.xlsx", "label_replacement", rows=12)

# 2b. Armonización y fusión de la reexportación PT3

### Propósito
Incorporar `data/raw/data_pt3/`: reexportaciones más recientes del CRM para proyecto03, proyecto04 y proyecto05 (mismo esquema de columnas "bonitas" que los Excel base proyecto01/proyecto02, sin el formato código+label de PT2) más un proyecto nuevo, proyecto06. PT2 tiene un problema de calidad conocido: `Fecha_Creación_Trato` queda constante (un mismo valor) en todas sus filas para proyecto03, proyecto04 y proyecto05, mientras que PT3 trae fechas reales.

### Regla de fusión con PT2
Para cada `id_Trato` presente en PT3 se conserva la fila de PT3 (fuente más reciente/confiable). Los `id_Trato` que sólo existen en PT2 se conservan sin cambios. Cuando los conjuntos de `id_Trato` de PT2 y PT3 coinciden por completo para un proyecto, el resultado equivale a usar sólo PT3; cuando difieren, el resultado es una unión (join). El detalle por proyecto se guarda en `reports/final/pt3_merge_summary.md`.

### Inputs
Archivos `data/raw/data_pt3/Reporte UVG *.xlsx` y el PT2 armonizado producido en la etapa 2.

### Outputs generados
CSV armonizados por proyecto y `pt3_all_harmonized.csv` en `data/raw/data_pt3/harmonized/`, el archivo fusionado `data/raw/data_pt2/harmonized/pt2_pt3_merged.csv` (input de la etapa 3) y `reports/final/pt3_merge_summary.md`.

In [ ]:
run_script("01b_harmonize_pt3_exports.py")

In [ ]:
pt3_merged_path = Path("data/raw/data_pt2/harmonized/pt2_pt3_merged.csv")
pt3_merged = load_csv(pt3_merged_path)
dataset_summary(pt3_merged, "PT2 + PT3 fusionados")
show_distribution(pt3_merged, "Proyecto")
show_distribution(pt3_merged, "Etapa Trato")

pt3_outputs = sorted(Path("data/raw/data_pt3/harmonized").glob("*.csv"))
pt3_outputs.append(pt3_merged_path)
artifact_inventory(pt3_outputs)
show_markdown_report("reports/final/pt3_merge_summary.md")

# 3. Construcción del dataset raw final

### Propósito
Crear una única base raw trazable que contenga tanto el historial consolidado de proyectos como las nuevas exportaciones armonizadas. En esta etapa el énfasis es la integración de datos, no el modelado predictivo.

### Inputs
Archivos históricos que coinciden con `data/raw/Leads UVG *.xlsx` y el archivo PT2+PT3 fusionado producido en la etapa 2b.

### Procesamiento realizado
`02_build_final_raw_dataset.py` alinea los esquemas, agrega la procedencia de cada fuente, concatena los registros y resuelve identificadores de negocio repetidos conservando la fila más completa. También comprueba que los archivos fuente base no hayan sido modificados y que todos los proyectos requeridos permanezcan representados.

### Outputs generados
`data/raw/leads_final_combined.csv`, el input raw para la limpieza posterior.


In [ ]:
run_script("02_build_final_raw_dataset.py")

In [ ]:
raw_final_path = Path("data/raw/leads_final_combined.csv")
raw_final = load_csv(raw_final_path)
dataset_summary(raw_final, "Dataset raw final")
show_distribution(raw_final, "Proyecto")
show_distribution(raw_final, "Etapa Trato")
artifact_inventory([raw_final_path])

# 4. Limpieza del dataset final

### Propósito
Convertir la tabla raw integrada en un dataset consistente y apto para modelado. La limpieza mejora la comparabilidad entre fuentes, controla observaciones duplicadas, define la variable objetivo del aprendizaje supervisado y elimina información que no estaría disponible en el momento de realizar una predicción.

### Inputs
`data/raw/leads_final_combined.csv` de la etapa 3 y las reglas reutilizables de `scripts/cleaning_helpers.py`.

### Procesamiento realizado
`03_clean_final_dataset.py` diagnostica la calidad de los datos; construye y valida `target_won`; elimina variables con fuga de información, con ausencia extrema o estructuralmente inadecuadas; normaliza texto categórico, campañas, fechas, edad, proyectos, datos de contacto y nombres de columnas; elimina registros de prueba; y deduplica negocios mediante la lógica de prioridad documentada.

### Outputs generados
Un dataset limpio apto para modelado, un dataset paralelo de auditoría con variables de fuga, una tabla que contiene únicamente variables predictoras, una referencia de fuga y reportes detallados de calidad, normalización, diccionario de variables, columnas eliminadas y resumen en Markdown.


In [ ]:
run_script("03_clean_final_dataset.py")

In [ ]:
clean_path = Path("data/processed/leads_cleaned_final.csv")
clean_with_leakage_path = Path("data/processed/leads_cleaned_with_leakage_final.csv")
clean = load_csv(clean_path)
clean_with_leakage = load_csv(clean_with_leakage_path)
validate_analytical_label_schema(clean, clean_path.name)
validate_analytical_label_schema(clean_with_leakage, clean_with_leakage_path.name)

dataset_summary(clean, "Dataset limpio apto para modelado")
show_distribution(clean, "target_won")
show_distribution(clean, "proyecto")
for variable in ["tiene_hijos", "sexo", "estado_civil"]:
    show_distribution(clean, variable, top=12)

comparison = pd.DataFrame(
    {
        "dataset": ["raw final", "limpio y apto para modelado", "auditoría limpia con fuga"],
        "filas": [len(raw_final), len(clean), len(clean_with_leakage)],
        "variables": [raw_final.shape[1], clean.shape[1], clean_with_leakage.shape[1]],
    }
)
display(Markdown("**Cambios de dimensiones durante la limpieza**"))
display(comparison)

example_columns = [col for col in ["id_trato", "proyecto", "campania", "edad", "tiene_hijos", "target_won"] if col in clean.columns]
display(Markdown("**Ejemplos de variables limpias**"))
display(clean[example_columns].head(10))

In [ ]:
cleaning_artifacts = [
    clean_path,
    clean_with_leakage_path,
    Path("data/processed/leads_features_final.csv"),
    Path("data/processed/leads_leakage_reference_final.csv"),
    Path("reports/final/data_quality_report.xlsx"),
    Path("reports/final/normalization_before_after.xlsx"),
    Path("reports/final/variable_dictionary.xlsx"),
    Path("reports/final/columns_removed.csv"),
    Path("reports/final/cleaning_summary.md"),
]
artifact_inventory(cleaning_artifacts)
show_markdown_report("reports/final/cleaning_summary.md")

# 5. Construcción de variables finales

### Propósito
Transformar los campos limpios en predictores interpretables y enriquecer los leads con contexto a nivel de proyecto. La ingeniería de variables proporciona representaciones estables para los algoritmos de aprendizaje automático y, al mismo tiempo, mantiene definiciones explícitas y auditables.

### Inputs
Los datasets limpios de la etapa 4, `scripts/feature_helpers.py`, la tabla de referencia de proyectos y los dos libros externos del mercado inmobiliario ubicados en `data/raw/`.

### Procesamiento realizado
`04_build_final_features.py` deriva la plataforma, el tipo y la audiencia de campaña; crea rangos de edad orientados al modelado; audita las fuentes externas; construye o sincroniza atributos de proyecto; enriquece tanto el dataset modelable como el dataset de auditoría con fuga; y ejecuta comprobaciones específicas de las variables creadas.

### Outputs generados
La tabla enriquecida final para modelado, su equivalente de auditoría, la referencia sincronizada de proyectos y los diagnósticos, mapeos, diccionarios y reportes Markdown de ingeniería de variables.


In [ ]:
run_script("04_build_final_features.py")

In [ ]:
enriched_path = Path("data/features/leads_enriched_final.csv")
enriched_audit_path = Path("data/features/leads_enriched_with_leakage_final.csv")
enriched = load_csv(enriched_path)
enriched_audit = load_csv(enriched_audit_path)
validate_analytical_label_schema(enriched, enriched_path.name)
validate_analytical_label_schema(enriched_audit, enriched_audit_path.name)

dataset_summary(enriched, "Dataset enriquecido final para modelado")
show_distribution(enriched, "target_won")
for variable in ["campania_plataforma", "campania_tipo", "campania_audiencia", "rango_edad_modelo", "zona_proyecto"]:
    show_distribution(enriched, variable, top=15)

new_features = [col for col in enriched.columns if col not in clean.columns]
display(Markdown(f"**Nuevas variables agregadas en la etapa 5 ({len(new_features)}):** `{'`, `'.join(new_features)}`"))
example_features = [col for col in ["proyecto", "campania", "campania_plataforma", "campania_tipo", "campania_audiencia", "edad", "rango_edad_modelo", "zona_proyecto", "precio_m2_promedio"] if col in enriched.columns]
display(Markdown("**Ejemplos de variables creadas y enriquecidas**"))
display(enriched[example_features].head(12))

In [ ]:
feature_artifacts = [
    enriched_path,
    enriched_audit_path,
    Path("data/reference/proyectos_atributos.csv"),
    Path("reports/final/fe_diagnostic_report.xlsx"),
    Path("reports/final/campania_mapping.xlsx"),
    Path("reports/final/fe_variable_dictionary_update.xlsx"),
    Path("reports/final/external_files_mapping.md"),
    Path("reports/final/fe_summary.md"),
]
artifact_inventory(feature_artifacts)
show_markdown_report("reports/final/fe_summary.md")

# 6. Validación de los outputs finales

### Propósito
Aplicar criterios explícitos de aceptación antes de utilizar el dataset para EDA o modelado. La validación convierte supuestos —como integridad del objetivo, conservación de filas, ausencia de duplicados y separación de variables con fuga— en comprobaciones ejecutables.

### Inputs
Todos los artefactos raw, limpios, de auditoría, enriquecidos y de resumen producidos en las etapas 3 a 5.

### Procesamiento realizado
`05_validate_final_outputs.py` comprueba la existencia y contenido de los archivos, los valores binarios del objetivo, la conservación de filas durante el enriquecimiento, los proyectos y variables obligatorios, la unicidad de identificadores de negocio, la clasificación de las etapas designadas como ganadas y la separación de variables con fuga de los archivos modelables.

### Outputs generados
Un registro visible de validación aprobada o fallida y `reports/final/eda_ready_summary.md`. Cualquier aserción fallida detiene el pipeline antes del EDA.


In [ ]:
run_script("05_validate_final_outputs.py")

In [ ]:
validation_observations = pd.DataFrame(
    {
        "comprobación": [
            "target_won contiene únicamente 0/1",
            "target_won no contiene valores faltantes",
            "id_trato es único",
            "las filas del dataset limpio y enriquecido coinciden",
            "las filas de auditoría limpia y enriquecida coinciden",
            "no existen columnas de código ni tipomodelo_label",
            "se conservan todas las columnas label requeridas",
        ],
        "observado": [
            set(enriched["target_won"].dropna().unique()).issubset({0, 1}),
            enriched["target_won"].notna().all(),
            not enriched["id_trato"].duplicated().any(),
            len(clean) == len(enriched),
            len(clean_with_leakage) == len(enriched_audit),
            not (FORBIDDEN_ANALYTICAL_COLUMNS & set(enriched.columns)),
            REQUIRED_ANALYTICAL_LABEL_COLUMNS.issubset(enriched.columns),
        ],
    }
)
display(validation_observations)
artifact_inventory([Path("reports/final/eda_ready_summary.md")])
show_markdown_report("reports/final/eda_ready_summary.md")

# 7. Ejecución del análisis exploratorio de datos final

### Propósito
Caracterizar el dataset validado y apto para modelado antes de aplicar clustering o aprendizaje supervisado. El EDA permite identificar el desbalance de clases, los valores faltantes, la cardinalidad, los rangos numéricos, el soporte de las categorías y posibles riesgos de modelado.

### Inputs
`data/features/leads_enriched_final.csv`, que excluye fuga posterior al resultado y aprobó la etapa 6.

### Procesamiento realizado
`06_run_final_eda.py` construye tablas de descripción general del dataset, variable objetivo, valores faltantes, cardinalidad, variables numéricas, asociaciones con el objetivo, alertas de soporte y recomendaciones. El script verifica que su input no haya sido modificado y no crea subconjuntos de entrenamiento.

### Outputs generados
`reports/eda/eda_technical_report.xlsx` y `reports/eda/eda_summary.md`. El notebook muestra hojas seleccionadas de estos reportes en lugar de recalcular el análisis.


In [ ]:
run_script("06_run_final_eda.py")

In [ ]:
eda_xlsx = Path("reports/eda/eda_technical_report.xlsx")
eda_md = Path("reports/eda/eda_summary.md")
artifact_inventory([eda_xlsx, eda_md])

display(Markdown("### Descripción general del dataset para EDA"))
show_excel_sheet(eda_xlsx, "dataset_overview", rows=30)
display(Markdown("### Balance de la variable objetivo"))
show_excel_sheet(eda_xlsx, "target_balance", rows=10)
display(Markdown("### Variables con mayor cantidad de valores faltantes"))
show_excel_sheet(eda_xlsx, "missingness", rows=15)
display(Markdown("### Recomendaciones del EDA"))
show_excel_sheet(eda_xlsx, "eda_recommendations", rows=30)

In [ ]:
show_markdown_report(eda_md)

# Registro de reproducibilidad y entrega

Una ejecución exitosa significa que las siete secciones terminaron sin excepciones y que la validación final informó que todas las comprobaciones fueron aprobadas. El principal input para modelado es:

- `data/features/leads_enriched_final.csv` — enriquecido, apto para modelado y libre de fuga de información posterior al resultado.

Los archivos equivalentes que contienen variables con fuga son artefactos de auditoría y no deben utilizarse como matrices de predictores. Los reportes técnicos en Excel y Markdown conservan evidencia detallada para la revisión metodológica, mientras que los scripts originales permanecen como la implementación autoritativa de cada transformación.


In [ ]:
final_record = pd.DataFrame(
    {
        "elemento": ["finalizado_el", "python", "dataset_apto_para_modelado", "filas", "variables", "faltantes_objetivo", "ids_duplicados"],
        "valor": [
            datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            sys.version.split()[0],
            enriched_path.as_posix(),
            len(enriched),
            enriched.shape[1],
            int(enriched["target_won"].isna().sum()),
            int(enriched["id_trato"].duplicated().sum()),
        ],
    }
)
display(final_record)
display(Markdown("## Pipeline completado exitosamente"))